# Notebook 05 — Protocole de Validation

> **Objectif** : construire un protocole de validation adapté aux données industrielles rares.  
> **Pourquoi c'est crucial** : un protocole inadapté (train/test classique) donnera des métriques trompeuses sur les variantes rares.

---

## Concepts abordés
1. Pourquoi le train/test split classique échoue sur données rares
2. Leave-One-Out Cross-Validation (LOO-CV)
3. Validation hiérarchique croisée (heldout families)
4. Validation temporelle glissante (walk-forward)
5. Métriques asymétriques pour le pricing industriel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
from itertools import combinations
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet(Path('data/dataset_industriel.parquet'))
FEATURES = ['composants', 'mo', 'energie', 'volume']
TARGET   = 'cout'
print(f"Dataset : {len(df):,} lignes, {df['variante'].nunique()} variantes")

## 1. Pourquoi le train/test split classique échoue

**Sur 5 observations avec un split 80/20** :
- 4 points en train, 1 point en test
- L'IC sur le RMSE calculé sur 1 point est infini — impossible de conclure
- Si la 1 observation de test est un outlier, le RMSE est catastrophique par hasard

**Démonstration empirique :**

In [ ]:
# Simulation : variabilité du RMSE test selon le split aléatoire
# On prend une variante rare (n=5) et on évalue le RMSE 1000 fois

TRUE_MEAN = 1000.0
SIGMA_OBS = 80.0
N_OBS = 5

rmse_tests = []
for _ in range(1000):
    y = np.random.normal(TRUE_MEAN, SIGMA_OBS, N_OBS)
    X = np.random.normal(0, 1, (N_OBS, 4))

    # Split 80/20 aléatoire
    idx_test = np.random.choice(N_OBS, size=1, replace=False)
    idx_train = np.setdiff1d(np.arange(N_OBS), idx_test)

    model = Ridge(alpha=1.0)
    model.fit(X[idx_train], y[idx_train])
    y_pred = model.predict(X[idx_test])
    rmse_tests.append(abs(y[idx_test[0]] - y_pred[0]))

print(f"RMSE test sur 1000 splits aléatoires (n=5) :")
print(f"  Médiane : {np.median(rmse_tests):.0f}€")
print(f"  P10     : {np.percentile(rmse_tests, 10):.0f}€")
print(f"  P90     : {np.percentile(rmse_tests, 90):.0f}€")
print(f"  → Plage de [P10; P90] = {np.percentile(rmse_tests, 90) - np.percentile(rmse_tests, 10):.0f}€")
print(f"    Conclusion : le RMSE test est inutilisable sur 1 seul point !")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rmse_tests, bins=50, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.axvline(np.median(rmse_tests), color='black', linestyle='--',
           lw=2, label=f'Médiane = {np.median(rmse_tests):.0f}€')
ax.axvline(SIGMA_OBS, color='#27ae60', linestyle='--',
           lw=2, label=f'Vraie σ = {SIGMA_OBS}€')
ax.set_xlabel('Erreur absolue (€)')
ax.set_ylabel('Fréquence sur 1000 splits')
ax.set_title('Distribution du RMSE test selon le split aléatoire\n'
             '(n=5 obs) — Variance énorme, métrique inutilisable')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_train_test_fail.png', dpi=150)
plt.show()

## 2. Leave-One-Out Cross-Validation (LOO-CV)

**Principe** : avec $n$ observations, on effectue $n$ entraînements : à chaque fois, on retire 1 observation pour la tester.

$$\text{LOO-RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_{-i})^2}$$

**Avantage** : utilise le maximum de données possible pour chaque évaluation.  
**Limites** : coûteux en calcul pour n grand ; biais légèrement optimiste (modèle entraîné sur n-1 points).

In [ ]:
def loo_cv(X: np.ndarray, y: np.ndarray, alpha: float = 10.0) -> float:
    """LOO-CV avec Ridge. Retourne le RMSE LOO."""
    n = len(y)
    if n < 3:
        return np.nan

    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    errors = []
    for i in range(n):
        X_tr = np.delete(X_std, i, axis=0)
        y_tr = np.delete(y, i)
        X_te = X_std[i:i+1]
        y_te = y[i]

        model = Ridge(alpha=alpha)
        model.fit(X_tr, y_tr)
        errors.append((y_te - model.predict(X_te)[0]) ** 2)

    return np.sqrt(np.mean(errors))


print("Calcul LOO-CV par variante...")
loo_scores = []

for variante, gdf in df.groupby('variante'):
    X_v = gdf[FEATURES].values
    y_v = gdf[TARGET].values
    n   = len(y_v)
    loo_scores.append({
        'variante': variante,
        'famille' : gdf['famille'].iloc[0],
        'n_obs'   : n,
        'loo_rmse': loo_cv(X_v, y_v),
    })

df_loo = pd.DataFrame(loo_scores)
df_loo['categorie'] = pd.cut(
    df_loo['n_obs'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print("\nLOO-RMSE par catégorie :")
print(df_loo.groupby('categorie')['loo_rmse'].describe().round(1))

## 3. Validation hiérarchique croisée (heldout families)

**Idée** : retirer une famille entière du train, puis prédire ses variantes.  
Cela teste la capacité de **généralisation inter-famille** — essentielle pour les vraies nouvelles variantes d'une nouvelle famille.

In [ ]:
le_famille = LabelEncoder().fit(df['famille'])
le_gamme   = LabelEncoder().fit(df['gamme'])

df_enc = df.copy()
df_enc['famille_enc'] = le_famille.transform(df['famille'])
df_enc['gamme_enc']   = le_gamme.transform(df['gamme'])
GLOBAL_FEATURES = FEATURES + ['famille_enc', 'gamme_enc']

familles = df['famille'].unique()
heldout_results = []

# On itère sur un sous-ensemble de familles pour rester rapide
for famille_test in familles[:8]:
    mask_test  = df_enc['famille'] == famille_test
    mask_train = ~mask_test

    X_tr = df_enc.loc[mask_train, GLOBAL_FEATURES].values
    y_tr = df_enc.loc[mask_train, TARGET].values
    X_te = df_enc.loc[mask_test,  GLOBAL_FEATURES].values
    y_te = df_enc.loc[mask_test,  TARGET].values

    scaler = StandardScaler()
    X_tr_std = scaler.fit_transform(X_tr)
    X_te_std = scaler.transform(X_te)

    model = Ridge(alpha=10.0)
    model.fit(X_tr_std, y_tr)
    y_pred = model.predict(X_te_std)

    heldout_results.append({
        'famille'      : famille_test,
        'n_test'       : mask_test.sum(),
        'rmse_heldout' : np.sqrt(mean_squared_error(y_te, y_pred)),
        'biais_moyen'  : (y_pred - y_te).mean(),
    })

df_heldout = pd.DataFrame(heldout_results)
print("Validation hiérarchique — Résultats par famille :")
print(df_heldout.round(1).to_string(index=False))

## 4. Métriques asymétriques pour le pricing industriel

Le RMSE est **symétrique** : sur-estimer de 100€ = sous-estimer de 100€.  
Or en pricing industriel, **sous-estimer le coût = marge négative** — bien plus grave.

On définit une **perte asymétrique** qui pénalise 3× plus les sous-estimations :

In [ ]:
def pricing_loss(y_true: np.ndarray, y_pred: np.ndarray,
                 penalty_under: float = 3.0) -> float:
    """
    Métrique asymétrique pour le pricing industriel.
    Pénalise `penalty_under`x plus les sous-estimations (marge négative).
    """
    error = y_pred - y_true
    loss = np.where(error < 0,
                    penalty_under * error**2,   # sous-estimation
                    error**2)                    # sur-estimation
    return np.sqrt(loss.mean())


def coverage_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """% de commandes où le prix prédit couvre le coût réel (y_pred >= y_true)."""
    return (y_pred >= y_true).mean()


def p90_absolute_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """90e percentile de l'erreur absolue (queue de distribution)."""
    return np.percentile(np.abs(y_pred - y_true), 90)


# Application sur les données
from sklearn.ensemble import GradientBoostingRegressor

X_all = df_enc[GLOBAL_FEATURES].values
y_all = df_enc[TARGET].values

scaler = StandardScaler()
X_std  = scaler.fit_transform(X_all)

model = Ridge(alpha=10.0)
model.fit(X_std, y_all)
y_pred_all = model.predict(X_std)

df_enc['pred'] = y_pred_all
df_enc['categorie'] = pd.cut(
    df_enc['n_obs_variante'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print("Métriques business par catégorie :")
print(f"{'Catégorie':<20} {'RMSE':>8} {'Asym.Loss':>12} {'Coverage':>10} {'P90 err':>10}")
print("-" * 65)
for cat in ['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']:
    g = df_enc[df_enc['categorie'] == cat]
    yt = g[TARGET].values
    yp = g['pred'].values
    print(f"{cat:<20} {np.sqrt(mean_squared_error(yt,yp)):>8.1f} "
          f"{pricing_loss(yt,yp):>12.1f} "
          f"{coverage_rate(yt,yp):>10.2%} "
          f"{p90_absolute_error(yt,yp):>10.1f}")

In [ ]:
# Visualisation de la perte asymétrique
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de la fonction de perte
ax = axes[0]
errors = np.linspace(-200, 200, 500)
loss_sym   = errors**2
loss_asym  = np.where(errors < 0, 3 * errors**2, errors**2)

ax.plot(errors, np.sqrt(loss_sym),  label='RMSE classique (symétrique)', color='#3498db', lw=2)
ax.plot(errors, np.sqrt(loss_asym), label='Perte asymétrique (3x sur sous-estim.)', color='#e74c3c', lw=2)
ax.axvline(0, color='gray', linestyle='--', lw=1)
ax.fill_betweenx([0, 200], -200, 0, alpha=0.1, color='#e74c3c', label='Zone sous-estimation (marge < 0)')
ax.set_xlabel('Erreur (y_pred - y_true) en €')
ax.set_ylabel('Perte')
ax.set_title('Fonction de perte : symétrique vs asymétrique')
ax.legend(fontsize=9)
ax.set_ylim(0, 350)

# Distribution des erreurs par catégorie
ax = axes[1]
df_enc['erreur'] = df_enc['pred'] - df_enc[TARGET]
palette = {'Rare (≤5)': '#e74c3c', 'Interméd. (6-24)': '#f39c12', 'Mature (≥25)': '#27ae60'}
for cat, color in palette.items():
    data = df_enc[df_enc['categorie'] == cat]['erreur']
    ax.hist(data, bins=30, alpha=0.5, color=color, label=cat)
ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.set_xlabel('Erreur (prédit - réel) en €')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des erreurs par catégorie\n(négatif = sous-estimation = risque)')
ax.legend()

plt.tight_layout()
plt.savefig('data/fig_asymmetric_loss.png', dpi=150)
plt.show()

## Résumé du Notebook 05 — Protocole recommandé

```
Niveau 1 — LOO-CV (obligatoire pour les variantes rares)
  ✓ Seul protocole statistiquement valide avec < 6 obs
  ✓ Utilise toutes les données disponibles

Niveau 2 — Validation hiérarchique (heldout families)
  ✓ Teste la généralisation inter-famille
  ✓ Crucial pour les nouvelles familles / cold start

Niveau 3 — Validation temporelle (walk-forward)
  ✓ Évite la fuite temporelle (inflation des prix, évolution des composants)
  ✓ Train sur M1-M18, prédire M19-M24
```

| Métrique | Utilité |
|----------|--------|
| RMSE | Évaluation globale (but : le réduire) |
| Perte asymétrique | Pénalise 3x les sous-estimations (marge négative) |
| Taux de couverture | % de commandes où le prix couvre le coût réel |
| P90 erreur absolue | Queue de distribution — erreurs extrêmes |
| Biais par famille | Dérive systématique sur certaines familles |

**→ Notebook suivant : [06_integration_expert_et_cold_start.ipynb](06_integration_expert_et_cold_start.ipynb)**